# 04 CSI Error and Clutter Robustness

This notebook visualizes robustness to:
- CSI estimation errors (`csi_error_std`)
- Sensing clutter variance (`clutter_scale`)

Input: `result/table/04_csi_clutter_robustness.csv`

Outputs:
- PDF figures in `result/figure/`
- Aggregated CSV tables in `result/table/`


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path(__file__).resolve().parents[1]
tbl_dir = ROOT / "result" / "table"
fig_dir = ROOT / "result" / "figure"
fig_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(tbl_dir / "04_csi_clutter_robustness.csv")
df.head()


In [ ]:
# Aggregate mean/std over runs
metrics = ["avg_sum_queue", "avg_sense_u", "deadline_violation_rate", "avg_slot_ms"]

agg = df.groupby(["method", "clutter_scale", "csi_error_std"])[metrics].agg(["mean", "std"]).reset_index()
agg.columns = ["method", "clutter_scale", "csi_error_std"] + [f"{m}_{s}" for m in metrics for s in ["mean", "std"]]

out_table = tbl_dir / "04_robustness_table_full.csv"
agg.to_csv(out_table, index=False)
print("Saved table:", out_table)

agg.head()


In [ ]:
# Line plots: queue performance vs CSI error, for each clutter level
methods = sorted(agg["method"].unique())
for cs in sorted(agg["clutter_scale"].unique()):
    plt.figure()
    sub_cs = agg[agg["clutter_scale"] == cs]
    for m in methods:
        sub = sub_cs[sub_cs["method"] == m].sort_values("csi_error_std")
        plt.errorbar(
            sub["csi_error_std"],
            sub["avg_sum_queue_mean"],
            yerr=sub["avg_sum_queue_std"],
            marker="o",
            capsize=3,
            label=m,
        )
    plt.xlabel("CSI error std (relative)")
    plt.ylabel("Average total queue")
    plt.title(f"Clutter scale = {cs:.1f}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    out = fig_dir / f"04_queue_vs_csi_clutter{cs:.1f}.pdf"
    plt.savefig(out, format="pdf", bbox_inches="tight")
    print("Saved:", out)


In [ ]:
# Line plots: sensing performance vs CSI error, for each clutter level
methods = sorted(agg["method"].unique())
for cs in sorted(agg["clutter_scale"].unique()):
    plt.figure()
    sub_cs = agg[agg["clutter_scale"] == cs]
    for m in methods:
        sub = sub_cs[sub_cs["method"] == m].sort_values("csi_error_std")
        plt.errorbar(
            sub["csi_error_std"],
            sub["avg_sense_u_mean"],
            yerr=sub["avg_sense_u_std"],
            marker="o",
            capsize=3,
            label=m,
        )
    plt.xlabel("CSI error std (relative)")
    plt.ylabel("Average sensing uncertainty $u_t$")
    plt.title(f"Clutter scale = {cs:.1f}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    out = fig_dir / f"04_sense_u_vs_csi_clutter{cs:.1f}.pdf"
    plt.savefig(out, format="pdf", bbox_inches="tight")
    print("Saved:", out)


In [ ]:
# Heatmap: TOAH gain over the best fixed hierarchy (lower is better).
# Gain is defined for the sum of two normalized metrics.

# Normalize across all (method, clutter, csi) points for a stable composite score.
q_all = agg["avg_sum_queue_mean"].values
u_all = agg["avg_sense_u_mean"].values
q_min, q_max = float(np.min(q_all)), float(np.max(q_all))
u_min, u_max = float(np.min(u_all)), float(np.max(u_all))

def composite_score(df_m: pd.DataFrame) -> np.ndarray:
    q = df_m["avg_sum_queue_mean"].values
    u = df_m["avg_sense_u_mean"].values
    qn = (q - q_min) / (q_max - q_min + 1e-12)
    un = (u - u_min) / (u_max - u_min + 1e-12)
    return qn + un

grid_csi = sorted(agg["csi_error_std"].unique())
grid_clutter = sorted(agg["clutter_scale"].unique())

gain = np.full((len(grid_clutter), len(grid_csi)), np.nan)

for iy, cs in enumerate(grid_clutter):
    for ix, csi in enumerate(grid_csi):
        sub = agg[(agg["clutter_scale"] == cs) & (agg["csi_error_std"] == csi)]
        toah = sub[sub["method"] == "TOAH"]
        fixed_s = sub[sub["method"] == "Fixed-SensingLeader"]
        fixed_c = sub[sub["method"] == "Fixed-CommLeader"]

        if len(toah) == 0 or len(fixed_s) == 0 or len(fixed_c) == 0:
            continue

        s_toah = composite_score(toah)[0]
        s_fixed = min(composite_score(fixed_s)[0], composite_score(fixed_c)[0])
        gain[iy, ix] = s_fixed - s_toah  # positive means TOAH improves

plt.figure()
im = plt.imshow(gain, aspect="auto", origin="lower")
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(grid_csi)), [f"{x:.2f}" for x in grid_csi])
plt.yticks(range(len(grid_clutter)), [f"{x:.1f}" for x in grid_clutter])
plt.xlabel("CSI error std (relative)")
plt.ylabel("Clutter scale")
plt.title("TOAH gain over best fixed hierarchy")
out = fig_dir / "04_heatmap_toah_gain.pdf"
plt.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out)


In [ ]:
# Compact table for the paper: pick a mid-condition and report mean±std.
mid = agg[(agg["clutter_scale"] == 2.0) & (agg["csi_error_std"] == 0.10)].copy()
mid = mid.sort_values("avg_sum_queue_mean")

def fmt(mean, std):
    return f"{mean:.3f} ± {std:.3f}"

mid_table = pd.DataFrame({
    "method": mid["method"],
    "avg_sum_queue": [fmt(m, s) for m, s in zip(mid["avg_sum_queue_mean"], mid["avg_sum_queue_std"])],
    "avg_sense_u": [fmt(m, s) for m, s in zip(mid["avg_sense_u_mean"], mid["avg_sense_u_std"])],
    "deadline_viol": [fmt(m, s) for m, s in zip(mid["deadline_violation_rate_mean"], mid["deadline_violation_rate_std"])],
    "avg_slot_ms": [fmt(m, s) for m, s in zip(mid["avg_slot_ms_mean"], mid["avg_slot_ms_std"])],
})
out_mid = tbl_dir / "04_table_mid_condition.csv"
mid_table.to_csv(out_mid, index=False)
print("Saved:", out_mid)

mid_table
